# Maritime Vessel Fuel Efficiency — ML & Optimization

Predict vessel fuel consumption and analyze how speed and operating conditions influence fuel efficiency.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path("vessel_fuel_data.csv")
OUTPUT_DIR = Path("maritime_fuel_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

## 1. Load and validate the vessel dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

required = [
    "speed_knots", "engine_power_kw", "draft_m",
    "cargo_tons", "distance_nm", "wind_speed_mps",
    "wave_height_m", "fuel_consumption_tpd"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.drop_duplicates().copy()
df = df[required].apply(pd.to_numeric, errors="coerce").dropna()

print("Shape:", df.shape)
display(df.head())

## 2. Exploratory analysis

In [ ]:
print(df.describe().T)

plt.figure(figsize=(9, 6))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation Matrix — Vessel Fuel Efficiency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x="speed_knots", y="fuel_consumption_tpd", alpha=0.5)
plt.title("Speed vs Fuel Consumption")
plt.xlabel("Speed (knots)")
plt.ylabel("Fuel Consumption (tons/day)")
plt.tight_layout()
plt.show()

## 3. Feature engineering

In [ ]:
df["power_per_cargo_ton"] = df["engine_power_kw"] / (df["cargo_tons"] + 1)
df["distance_per_day_proxy"] = df["speed_knots"] * 24
df["weather_index"] = df["wind_speed_mps"] + 2 * df["wave_height_m"]

features = [
    "speed_knots", "engine_power_kw", "draft_m",
    "cargo_tons", "distance_nm", "wind_speed_mps",
    "wave_height_m", "power_per_cargo_ton",
    "distance_per_day_proxy", "weather_index"
]

target = "fuel_consumption_tpd"

## 4. Train the fuel-consumption model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=0.20,
    random_state=RANDOM_STATE
)

model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

## 5. Evaluate model performance

In [ ]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

metrics = pd.DataFrame([{
    "MAE_tpd": mae,
    "RMSE_tpd": rmse,
    "R2": r2
}])

display(metrics.round(4))

plt.figure(figsize=(7, 7))
plt.scatter(y_test, pred, alpha=0.5)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, "--")
plt.xlabel("Actual Fuel Consumption")
plt.ylabel("Predicted Fuel Consumption")
plt.title("Actual vs Predicted Fuel Consumption")
plt.tight_layout()
plt.show()

## 6. Feature importance

In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x="importance", y="feature")
plt.title("Fuel Consumption Driver Importance")
plt.tight_layout()
plt.show()

## 7. Speed-scenario simulation

In [ ]:
base = df.median(numeric_only=True).to_dict()

scenarios = []
for speed in [10, 11, 12, 13, 14, 15, 16]:
    row = base.copy()
    row["speed_knots"] = speed
    row["power_per_cargo_ton"] = row["engine_power_kw"] / (row["cargo_tons"] + 1)
    row["distance_per_day_proxy"] = speed * 24
    row["weather_index"] = row["wind_speed_mps"] + 2 * row["wave_height_m"]
    X_scenario = pd.DataFrame([row])[features]
    predicted = model.predict(X_scenario)[0]
    scenarios.append({
        "speed_knots": speed,
        "predicted_fuel_tpd": predicted,
        "fuel_per_nm": predicted / (speed * 24)
    })

scenarios = pd.DataFrame(scenarios)
display(scenarios)

plt.figure(figsize=(9, 5))
plt.plot(scenarios["speed_knots"], scenarios["predicted_fuel_tpd"], marker="o")
plt.title("Predicted Fuel Consumption by Speed")
plt.xlabel("Speed (knots)")
plt.ylabel("Predicted Fuel (tons/day)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Export outputs

In [ ]:
predictions = X_test.copy()
predictions["actual_fuel_tpd"] = y_test.values
predictions["predicted_fuel_tpd"] = pred

predictions.to_csv(OUTPUT_DIR / "fuel_predictions.csv", index=False)
scenarios.to_csv(OUTPUT_DIR / "efficiency_scenarios.csv", index=False)

with pd.ExcelWriter(OUTPUT_DIR / "maritime_fuel_analysis.xlsx", engine="openpyxl") as writer:
    metrics.to_excel(writer, sheet_name="Model Metrics", index=False)
    importance.to_excel(writer, sheet_name="Feature Importance", index=False)
    predictions.to_excel(writer, sheet_name="Predictions", index=False)
    scenarios.to_excel(writer, sheet_name="Speed Scenarios", index=False)

print("Outputs saved to:", OUTPUT_DIR)